In [ ]:
import os
from pathlib import Path

import pandas as pd
import xarray as xr
import rasterio
from rasterio.transform import from_origin

The aim of this script is to create a monthly raster from ERA5 netcdf and save it as a geotiff. 
Parameters: total precipitation, mean temperature, minimum temperature, maximum temperature.

Two kinds of netcdfs are used: monthly and hourly, both consist of one parameter and one month's data of a specific year. The hourly 2m temperature rasters are aggregated into mean, min, and max rasters. The monthly temperature raster represents mm of precipitation for a day in that month, so the data is multiplied by number of days in that month.

In [ ]:
# Define file paths

# working directory
wd = os.getcwd() 

os.chdir("..\\") 

# path to .nc folder
netcdf_folder = os.getcwd() + "\\scripts\\era5_input_rasters\\"

# Temporary local path for GeoTIFFs before upload to bucket
tiff_temp_path = f"{netcdf_folder}\\era5_monthly_rasters_2020_2023\\"  

netcdf_folder

In [ ]:
# Define geotiff file name prefixes

region_name = 'est'

provider = 'era5'

temporal_res = 'monthly'

In [ ]:
# Specify the parameters and dates

years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

months = list(range(1, 13, 1)) # all months of the year

# parameters we want to have
parameters = ['total_precipitation_sum', \
              'mean_2m_temperature', 'minimum_2m_temperature', 'maximum_2m_temperature', 'total_evaporation_sum']

In [ ]:
### functions

In [ ]:
def create_subfolders():
    """Creates the local subfolders where the geotiffs will be stored: 
    tiff_temp_path/{parameter}/{year}
    for the provided parameters and years.
    Ignores if subfolders exist."""
    
    for year in years:
        
        for parameter in parameters:
            
            
            full_subfolder_path = f"{tiff_temp_path}\\{parameter}\\{year}"   
            
            Path(full_subfolder_path).mkdir(parents=True, exist_ok=True)
            

In [ ]:
def netcdf_to_gtiffs(netcdf_path):
    """Opens the netcdf, saves the metadata, 
    calls export based on the parameter"""
    # Open the NetCDF file
    with xr.open_dataset(netcdf_path) as ds:
        
        # get variables and time slices
        variable_name = list(ds.data_vars)[0] # works if it is the only one!
        
        # year and month of the dataset
        date = ds[variable_name].coords['valid_time'].values[0] # save the date for filename
        
        
        # Get the transform
        latitudes = ds['latitude'].values
        
        longitudes = ds['longitude'].values
        
        transform = from_origin(longitudes.min(), latitudes.max(), 
                                abs(longitudes[1] - longitudes[0]), 
                                abs(latitudes[1] - latitudes[0]))
        
        # For some reason, the variable names between GEE and Copernicus
        # service differ. Added this to keep the parameter names consistent.
        # Not sure if it matters though...
        if variable_name == 't2m': # temperature
            
            # creating and exporting a mean, min, and max raster
            mean_raster, max_raster, min_raster = create_monthly_raster(ds, variable_name)
            
            export_time_slices(mean_raster, date, transform, variable_name, 'mean_2m_temperature')
            
            export_time_slices(max_raster, date, transform, variable_name, 'maximum_2m_temperature')
            
            export_time_slices(min_raster, date, transform, variable_name, 'minimum_2m_temperature')
                
    
        elif variable_name == 'tp': # precipitation

            parameter = 'total_precipitation_sum'
            
            export_time_slices(ds, date, transform, variable_name, parameter)
            
        elif variable_name == 'e': # evaporation
                
            parameter = 'total_evaporation_sum'
            
            export_time_slices(ds, date, transform, variable_name, parameter)
        
        
        else: # Warning that the name from .nc is used in filename
               
            print(f"Parameter name {variable_name} from .nc used in filename. May differ in GEE.")
            
            
            parameter = variable_name
            
            export_time_slices(ds, date, transform, variable_name, parameter)
                

In [ ]:
def export_time_slices(ds, date, transform, variable_name, parameter):       
        """Exports provided dataset as a geotiff"""
  
        # Define GeoTIFF filename
        tiff_filename = f"{region_name}_{provider}_{parameter}_{temporal_res}_{str(date)[0:4]}_{str(date)[5:7]}.tif"
        
        print(tiff_filename)
        local_tiff_path = f"{tiff_temp_path}\\{parameter}\\{str(date)[0:4]}\\{tiff_filename}"
            
            
        # In ERA5 monthly rasters, the precipitation is m in a day in that month, so need to multiply
        # by number of days in that month
        if parameter == 'total_precipitation_sum' or parameter == 'total_evaporation_sum':

            # remove spare dimensions (indicates 4, but 2 are needed, i.e., lat and lon)
            ds = ds.squeeze()
            
            # retrieve the days in month on the fly
            days_in_month = pd.Timestamp(date).days_in_month
            # multiply each value by days in that month
            ds[variable_name] = ds[variable_name] * days_in_month  
                
        # Set up metadata for the GeoTIFF
        metadata = {
                'driver': 'GTiff',
                'height': ds[variable_name].shape[0],
                'width': ds[variable_name].shape[1],
                'count': 1,
                'dtype': str(ds[variable_name].dtype),
                'crs': 'EPSG:4326',
                'transform': transform
            }
            
           
        # Write the raster data to a GeoTIFF
        with rasterio.open(local_tiff_path, 'w', **metadata) as dst:
                
            dst.write(ds[variable_name].values, 1)

In [ ]:
def create_monthly_raster(ds, variable_name):
    """ For the temperature .ncs, creates a mean, min, max of the month
    from the hourly timeslices"""
    
    # Compute the mean, min, max over the time dimension
    monthly_mean = ds[variable_name].mean(dim='valid_time')
    
    monthly_max = ds[variable_name].max(dim='valid_time')

    monthly_min = ds[variable_name].min(dim='valid_time')
    
    
    coordinates = {'latitude': ds['latitude'],  
            'longitude': ds['longitude']}
    
    
    # Create a new dataset with the computed monthly means
    mean_ds = xr.Dataset(
        {variable_name: monthly_mean},
        coords=coordinates,
        attrs=ds.attrs  # Preserve global metadata
    )
    
    max_ds = xr.Dataset(
        {variable_name: monthly_max},
        coords= coordinates,
        attrs=ds.attrs  # Preserve global metadata
    )
    
    min_ds = xr.Dataset(
        {variable_name: monthly_min},
        coords= coordinates,
        attrs=ds.attrs  # Preserve global metadata
    )
    
    return mean_ds, max_ds, min_ds

In [ ]:
### function calls

In [ ]:
create_subfolders()

In [ ]:
for file in os.listdir(netcdf_folder):
    
    if file.endswith(".nc"):
        
        print(file)

        full_path = netcdf_folder + file
        
        netcdf_to_gtiffs(full_path)
        